In [ ]:
import pandas as pd

summary_path = "neural_op_summary.csv"  # or "all_results.csv"

df = pd.read_csv(summary_path)

# Make sure numeric columns are actually numeric
numeric_cols = [
    "Noise Level",
    "MSE Loss",
    "Relative L2 Loss",
    "Physics Loss",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Add degradation columns per model/run_folder
df = df.sort_values(["run_folder", "Noise Level"]).reset_index(drop=True)

# Get baseline error at noise_level == 0 for each model
baseline = (
    df[df["Noise Level"] == 0.0]
    .set_index("run_folder")[["MSE Loss", "Relative L2 Loss"]]
    .rename(columns={
        "MSE Loss": "MSE Loss baseline",
        "Relative L2 Loss": "Relative L2 Loss baseline",
    })
)

# Attach baseline values to every row
df = df.join(baseline, on="run_folder")

# Compute degradation
df["MSE Degradation"] = (
    (df["MSE Loss"] - df["MSE Loss baseline"])
    / df["MSE Loss baseline"]
)

df["Relative L2 Degradation"] = (
    (df["Relative L2 Loss"] - df["Relative L2 Loss baseline"])
    / df["Relative L2 Loss baseline"]
)

# Optional: remove helper baseline columns
df = df.drop(columns=["MSE Loss baseline", "Relative L2 Loss baseline"])

models = {}

for run_folder, group in df.groupby("run_folder"):
    group = group.sort_values("Noise Level").reset_index(drop=True)
    models[run_folder] = {
        "result_folder": group["result_folder"].iloc[0],
        "metrics": {
            "noise_level": group["Noise Level"].to_numpy(),
            "mse": group["MSE Loss"].to_numpy(),
            "relative_l2": group["Relative L2 Loss"].to_numpy(),
            "physics": group["Physics Loss"].to_numpy(),
            "mse_degradation": group["MSE Degradation"].to_numpy(),
            "relative_l2_degradation": group["Relative L2 Degradation"].to_numpy(),
        },
        "df": group,
    }
# df.to_csv(summary_path.replace(".csv", "_with_degradation.csv"), index=False)

: 

In [ ]:
for run_folder, data in models.items():
    print(f"Model: {run_folder}")
    print(data["df"][["Noise Level", "MSE Loss", "Relative L2 Loss", "MSE Degradation", "Relative L2 Degradation", "Physics Loss"]])
    print("\n")
    break


: 

In [ ]:
import matplotlib.pyplot as plt
def plot_metric(models, metric_key, ylabel, title):
    plt.figure(figsize=(12, 5))

    for run_folder, data in models.items():
        plt.plot(
            data["metrics"]["noise_level"][:-2],
            data["metrics"][metric_key][:-2],
            marker="o",
            label=run_folder,
        )

    plt.xlabel("Noise Level")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.show()
    
def plot_metric_log(models, metric_key, ylabel, title):
    plt.figure(figsize=(12, 5))

    for run_folder, data in models.items():
        plt.plot(
            data["metrics"]["noise_level"],
            1 + data["metrics"][metric_key],  # Shift to avoid log(0)
            marker="o",
            label=run_folder,
        )

    plt.xlabel("Noise Level")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.yscale("log")
    plt.grid(True, which="both")
    plt.legend()
    plt.show()
plot_metric(models, "mse_degradation", "MSE Degradation", "MSE Degradation vs Noise Level")
plot_metric(models, "relative_l2_degradation", "Relative L2 Degradation", "Relative L2 Degradation vs Noise Level")

: 

In [ ]:
plot_metric(models, "mse", "MSE Loss", "MSE Loss vs Noise Level")
plot_metric(models, "relative_l2", "Relative L2 Loss", "Relative L2 Loss vs Noise Level")

: 

In [ ]:
plot_metric_log(models, "mse", "MSE Loss (log scale)", "MSE Loss vs Noise Level (Log Scale)")

: 

In [ ]:
plot_metric(models, "physics", "Physics Loss", "Physics Loss vs Noise Level")

: 